In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject 
from pyspark.sql import SparkSession
from lms_internal import LMSInternal
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_sales_payment_lms(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, lms_internal: LMSInternal
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "ExposureSalesPaymentId	string,"
        # "SourceSystem	string,"
        # "IngestionTimestamp	timestamp,"
        # "BatchId	string,"
        # "ExposureSalesPaymentKey	int,"
        # "SalesKey	int,"
        # "ExposureSalesDescription	string,"
        # "ExposureSalesPaymentTypeCode	string,"
        # "ExposureSalesPaymentType	string,"
        # "PaymentAmount	decimal(19,4),"
        # "PaymentDate	timestamp,"
        # "FinalSaleDate	timestamp,"
        # "TUN	string,"
        # "CreatedDate	timestamp,"
        # "CreatedBy	string,"
        # "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExposureSalesPayment"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "lms",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== loan_sales_payment ==================

        loan_sales_payment_extra_filters = None

        if loan_sales_payment_extra_filters is not None:
            loan_sales_payment_extra_filters.append(date_filter)
            loan_sales_payment_final_filters = " AND ".join(
                loan_sales_payment_extra_filters
            )
        else:
            loan_sales_payment_final_filters = date_filter

        loan_sales_payment_selected_fields = [
            "id",
            "loansalesFk",
            "type",
            "typedescription",
            "amount",
            "paymentdate",
            "finalSaleDate",
            "TUN",
            "CreationDate",
            "CreationUser",
            "businessdate",
        ]

        loan_sales_payment = lms_internal.loanSalesPayment()
        loan_sales_payment = loan_sales_payment.select(loan_sales_payment_selected_fields)

        # ================== loan_sales ==================

        loan_sales_extra_filters = None

        if loan_sales_extra_filters is not None:
            loan_sales_extra_filters.append(date_filter)
            loan_sales_final_filters = " AND ".join(loan_sales_extra_filters)
        else:
            loan_sales_final_filters = date_filter

        loan_sales_selected_fields = ["description", "id"]
        loan_sales = lms_internal.loanSales()
        loan_sales = loan_sales.select(loan_sales_selected_fields).withColumnRenamed("id", "id_ls")

        # ================== exposure_sales_payment_lms_df ==================

        exposure_sales_payment_lms_primary_key_fields = [
            loan_sales_payment.id,
            lit("lms"),
        ]

        exposure_sales_payment_lms_df = loan_sales_payment.join(
            loan_sales,
            (
                loan_sales_payment["LoanSalesFK"] == loan_sales["id_ls"]
            ),  # Change to inner join based on mapping (Excel)
            "inner",
        )

        exposure_sales_payment_lms_df = exposure_sales_payment_lms_df.withColumn(
            "ExposureSalesPaymentId",
            sha2(concat_ws(";", *exposure_sales_payment_lms_primary_key_fields), 256),
        )

        exposure_sales_payment_lms_df_final = exposure_sales_payment_lms_df.select(
            exposure_sales_payment_lms_df.ExposureSalesPaymentId.cast("string").alias("ExposureSalesPaymentId"),
            lit("lms").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).alias("BatchID"),
            loan_sales_payment.id.alias("ExposureSalesPaymentKey"),
            loan_sales_payment.loansalesFk.alias("SalesKey"),
            exposure_sales_payment_lms_df.description.cast("string").alias("ExposureSalesDescription"),
            exposure_sales_payment_lms_df.type.cast("string").alias("ExposureSalesPaymentTypeCode"),
            exposure_sales_payment_lms_df.typedescription.cast("string").alias("ExposureSalesPaymentType"),
            exposure_sales_payment_lms_df.amount.alias("PaymentAmount"),
            exposure_sales_payment_lms_df.paymentdate.alias("PaymentDate"),
            exposure_sales_payment_lms_df.finalSaleDate.alias("FinalSaleDate"),
            exposure_sales_payment_lms_df.TUN.alias("TUN"),
            exposure_sales_payment_lms_df.CreationDate.alias("CreatedDate"),
            exposure_sales_payment_lms_df.CreationUser.alias("CreatedBy"),
            loan_sales_payment.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        exposure_sales_payment_lms_df_final.write.format("delta").clusterBy(
            "BusinessDate"
        ).mode("append").saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
lms_internal = LMSInternal(spark, catalog_landed, run_date)

In [0]:
load_curated_exposure_sales_payment_lms(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, lms_internal)